In [9]:
from datasets import load_dataset, DatasetDict
import pandas as pd
# Load dataset from a CSV file
x = pd.read_csv("text_label.csv")
# Rename the 'name' column to 'full_name'
x.rename(columns={"Decision": "label"}, inplace=True)

# Save the updated DataFrame to a new CSV file
x.to_csv("label.csv", index=False)

# Verify the change
print(x.head())

                                                Text  label
0  Under review as a conference paper at ICLR 202...      0
1  Under review as a conference paper at ICLR 202...      0
2  Under review as a conference paper at ICLR 202...      0
3  Under review as a conference paper at ICLR 202...      0
4  Under review as a conference paper at ICLR 202...      0


In [3]:
train_testvalid = dataset["train"].train_test_split(test_size=0.4, shuffle=True, seed=42)

# Then, split the 40% into 20% test and 20% validation
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, shuffle=True, seed=42)

# Combine into a DatasetDict
dataset = DatasetDict({
    "train": train_testvalid["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
})

# Inspect the dataset
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Text', 'labels'],
        num_rows: 2344
    })
    validation: Dataset({
        features: ['Text', 'labels'],
        num_rows: 782
    })
    test: Dataset({
        features: ['Text', 'labels'],
        num_rows: 782
    })
})


In [4]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import KFold
import numpy as np

# Load dataset from a CSV file
dataset = load_dataset("csv", data_files="label.csv")

# Convert the dataset to a format compatible with KFold
data = dataset["train"]  # Access the dataset split
texts = data["Text"]     # Features (e.g., text)
labels = data["labels"]   # Labels (e.g., classification labels)

# Initialize KFold with 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate over the folds
for fold, (train_idx, val_idx) in enumerate(kf.split(texts)):
    print(f"Fold {fold + 1}")

    # Split the data into training and validation sets for this fold
    train_texts = np.array(texts)[train_idx].tolist()
    train_labels = np.array(labels)[train_idx].tolist()

    val_texts = np.array(texts)[val_idx].tolist()
    val_labels = np.array(labels)[val_idx].tolist()

    # Create Dataset objects for this fold
    train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
    val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})

    # Print fold statistics
    print(f"Train size: {len(train_dataset)}")
    print(f"Validation size: {len(val_dataset)}")
    print("---")

    # Here, you can train and evaluate your model for this fold
    # Example:
    # model.train(train_dataset)
    # model.evaluate(val_dataset)

Fold 1
Train size: 3126
Validation size: 782
---
Fold 2
Train size: 3126
Validation size: 782
---
Fold 3
Train size: 3126
Validation size: 782
---
Fold 4
Train size: 3127
Validation size: 781
---
Fold 5
Train size: 3127
Validation size: 781
---


In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load SciBERT tokenizer and model
model_name = "allenai/scibert_scivocab_uncased"  # SciBERT model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # 2 labels for binary classification

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
from datasets import load_dataset

# Load dataset from a CSV file
dataset = load_dataset("csv", data_files="label.csv")

# Access the dataset split
data = dataset["train"]

In [3]:
# Start with 10 rows
subset_size = 2
small_dataset = data.select(range(subset_size))

# Inspect the small dataset
print(small_dataset)

Dataset({
    features: ['Text', 'label'],
    num_rows: 2
})


In [4]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import KFold
import numpy as np
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

In [4]:
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["Text"],  # Replace "text" with the name of your text column
        padding="max_length",  # Pad sequences to the max_length
        truncation=True,       # Truncate sequences longer than max_length
        max_length=512,        # Set a fixed max_length (e.g., 512 for BERT-based models)
    )

# Apply the tokenizer to the dataset
tokenized_dataset = small_dataset.map(tokenize_function, batched=True)

# Inspect the tokenized dataset
print(tokenized_dataset)
print(tokenized_dataset[0])  # Inspect the first example

# Debug the tokenizer on a single example
example = small_dataset[0]["Text"]
tokenized_example = tokenizer(example, padding="max_length", truncation=True, max_length=512)
print(tokenized_example)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['Text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2
})
{'Text': 'Under review as a conference paper at ICLR 2021\nA\nGENERALIZED\nPROBABILITY\nKERNEL\nON\nDIS-\nCRETE DISTRIBUTIONS AND ITS APPLICATION IN TWO-\nSAMPLE TEST\nAnonymous authors\nPaper under double-blind review\nABSTRACT\nWe propose a generalized probability kernel(GPK) on discrete distributions with\nﬁnite support. This probability kernel, deﬁned as kernel between distributions in-\nstead of samples, generalizes the existing discrepancy statistics such as maximum\nmean discrepancy(MMD) as well as probability product kernels, and extends to\nmore general cases. For both existing and newly proposed statistics, we estimate\nthem through empirical frequency and illustrate the strategy to analyze the re-\nsulting bias and convergence bounds. We further propose power-MMD, a natural\nextension of MMD in the framework of GPK, illustrating its usage for the task\nof two-sample

In [5]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import KFold
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# Load dataset from a CSV file
dataset = load_dataset("csv", data_files="label.csv")

# Convert the dataset to a format compatible with KFold
data = dataset["train"]  # Access the dataset split

# Select only the first 10 data points
data = data.select(range(10))  # Slice the dataset to include only 10 rows
texts = data["Text"]           # Features (e.g., text)
labels = data["label"]         # Labels (e.g., classification labels)

# Initialize KFold with 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate over the folds
for fold, (train_idx, val_idx) in enumerate(kf.split(texts)):
    print(f"Fold {fold + 1}")

    # Split the data into training and validation sets for this fold
    train_texts = np.array(texts)[train_idx].tolist()
    train_labels = np.array(labels)[train_idx].tolist()

    val_texts = np.array(texts)[val_idx].tolist()
    val_labels = np.array(labels)[val_idx].tolist()

    # Create Dataset objects for this fold
    train_dataset = Dataset.from_dict({"Text": train_texts, "label": train_labels})
    val_dataset = Dataset.from_dict({"Text": val_texts, "label": val_labels})

    # Tokenize the datasets using SciBERT
    tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
    def tokenize_function(examples):
        return tokenizer(
            examples["Text"],
            padding="max_length",  # Pad sequences to max_length
            truncation=True,       # Truncate sequences longer than max_length
            max_length=512,        # Set a fixed max_length
        )

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)

    # Load the SciBERT model
    model = AutoModelForSequenceClassification.from_pretrained("allenai/scibert_scivocab_uncased", num_labels=2)

    # Set up training arguments
    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold + 1}",
        evaluation_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,  # Reduce batch size for small dataset
        per_device_eval_batch_size=4,   # Reduce batch size for evaluation
        num_train_epochs=3,
        weight_decay=0.01,
        save_strategy="epoch",
        logging_dir=f"./logs_fold_{fold + 1}",
        logging_steps=10,
    )

    # Define the Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
    )

    # Train the model
    trainer.train()

    # Evaluate the model
    eval_results = trainer.evaluate()
    print(f"Validation results for fold {fold + 1}: {eval_results}")

    # Save the model
    model.save_pretrained(f"./scibert_model_fold_{fold + 1}")
    tokenizer.save_pretrained(f"./scibert_model_fold_{fold + 1}")

Fold 1


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-5-0f3b8d4b4d81>:71: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer 

Epoch,Training Loss,Validation Loss
1,No log,0.192742
2,No log,0.133380
3,No log,0.117983


...

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-5-0f3b8d4b4d81>:71: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,0.196341
2,No log,0.091058
3,No log,0.069950


Validation results for fold 5: {'eval_loss': 0.06995002180337906, 'eval_runtime': 3.7236, 'eval_samples_per_second': 0.537, 'eval_steps_per_second': 0.269, 'epoch': 3.0}
